# Week 10: Beat the Baseline — Neural Network Training & Diagnostics

**Challenge:** Build a neural network to classify imagined hand movements (left vs right) from EEG brain signals. Compare it to a logistic regression baseline.

**Dataset:** Pre-extracted EEG features from 109 participants (PhysioNet EEGBCI)

See the [challenge brief](README.md) for full instructions and starter prompts.

---

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

# PyTorch — new this week!
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
print(f'PyTorch version: {torch.__version__}')
print('All imports successful!')

In [ ]:
# Cell 2: Load the data
data = pd.read_csv('data/eeg_motor_imagery_features.csv')

print(f'Dataset: {data.shape[0]:,} trials × {data.shape[1]} columns')
print(f'Participants: {data["participant_id"].nunique()}')
print()
data.head()

In [ ]:
# Cell 3: Explore class balance
print('Class balance:')
print(data['condition'].value_counts())
print()

fig, ax = plt.subplots(figsize=(6, 4))
data['condition'].value_counts().plot(kind='bar', color=['#4A90D9', '#A71930'], ax=ax)
ax.set_title('Class Balance: Left vs Right Motor Imagery')
ax.set_ylabel('Number of trials')
ax.set_xticklabels(['Left hand', 'Right hand'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Explore features
feature_cols = [c for c in data.columns
                if c not in ['participant_id', 'trial', 'condition', 'condition_code']]
print(f'Number of features: {len(feature_cols)}')
print(f'Feature names (first 10): {feature_cols[:10]}')
print()

# Summary statistics
print(data[feature_cols].describe().round(4))

In [ ]:
# Cell 5: Prepare data — split and standardise
X = data[feature_cols].values
y = data['condition_code'].values

# Stratified 80/20 split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,} trials')
print(f'Test:  {len(X_test):,} trials')

# Standardise features (CRITICAL for neural networks)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Use train stats only!

print(f'\nFeatures standardised:')
print(f'  Train mean: {X_train_scaled.mean():.6f} (should be ~0)')
print(f'  Train std:  {X_train_scaled.std():.6f} (should be ~1)')

---

## Step 1: Create Your Plan

Before writing any modelling code, ask your AI to create an analysis plan.

Use the **planning prompt** from the challenge brief. Save the plan as `plan.md`.

---

## Phase 1: Logistic Regression Baseline

This should feel familiar from Weeks 4 and 6. Fit a logistic regression classifier and report accuracy + F1.

**Your target to beat:** Whatever accuracy the baseline achieves, the neural network should do at least this well.

In [ ]:
# Cell 8: Logistic Regression Baseline
# TODO: Your AI will help you write this
#
# 1. Fit LogisticRegression(max_iter=1000, random_state=42) on X_train_scaled
# 2. Predict on X_test_scaled
# 3. Report accuracy and F1
# 4. Print classification report
#
# Expected: ~60% accuracy


---

## Phase 2: Build a Neural Network in PyTorch

This is new territory! You'll need to:
1. Convert data to PyTorch tensors
2. Define a network architecture
3. Write a training loop
4. Track and plot the loss

Use the **code prompt** from the challenge brief, or have your AI guide you step by step.

In [ ]:
# Cell 10: Convert to PyTorch tensors and create a DataLoader
# Your AI will help you fill this in.
#
# Convert the scaled numpy arrays to FloatTensors (features) and the labels
# to LongTensors (CrossEntropyLoss needs integer class labels).
# Then wrap the training tensors in a TensorDataset and a DataLoader so we
# can iterate in mini-batches (batch_size=32, shuffle=True) — this is the
# inner "batch loop" from the Week 9 lecture.
#
# Why shuffle? So each epoch sees the trials in a different order.
# Why batch_size=32? Standard small-batch default; small enough to be noisy
# (which helps the optimiser escape shallow valleys), big enough to use the
# CPU efficiently. Feel free to experiment in the bonus challenges.

In [ ]:
# Cell 11: Define the neural network
# Your AI will help you write a small MLP. Suggested shape:
#   Linear(n_features, 64) → ReLU → Dropout(0.3) → Linear(64, 2)
# Either as an nn.Module subclass (more flexible) or via nn.Sequential.
#
# Before creating the model, set torch.manual_seed(42) so your results are
# reproducible — neural networks start from random weights, so without a seed
# you'll get slightly different numbers every run.

In [ ]:
# Cell 12: Training loop (nested loops — exactly like the Week 9 flow chart)
# Your AI will help you write this. Suggested structure:
#
#   criterion = nn.CrossEntropyLoss()
#   optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#   train_losses, val_losses = [], []
#
#   for epoch in range(50):                       # outer EPOCH loop
#       model.train()
#       running = 0.0
#       for X_batch, y_batch in train_loader:     # inner BATCH loop
#           optimizer.zero_grad()
#           outputs = model(X_batch)
#           loss = criterion(outputs, y_batch)
#           loss.backward()
#           optimizer.step()
#           running += loss.item()
#       train_losses.append(running / len(train_loader))
#
#       # Evaluate on the test set each epoch (NO gradients — just measure)
#       model.eval()
#       with torch.no_grad():
#           val_loss = criterion(model(X_test_tensor), y_test_tensor)
#           val_losses.append(val_loss.item())
#
# Tracking BOTH losses is essential — the train/val gap is how you spot
# overfitting (Week 9 lecture).

---

## Training Diagnostics

Plot the training loss curve. This is your most important diagnostic:
- **Decreasing:** The network is learning
- **Flat at ~0.693:** The network is guessing randomly (ln(2) = 0.693)
- **Oscillating wildly:** Learning rate is too high

In [ ]:
# Cell 14: Plot the loss curves — your most important diagnostic
# Your AI will help you write this.
#
# Plot BOTH train_losses and val_losses on the same axes:
#   plt.plot(train_losses, label='Training loss')
#   plt.plot(val_losses,   label='Validation loss')
#   plt.axhline(y=np.log(2), color='gray', linestyle='--',
#               label=f'Random guessing (ln 2 ≈ 0.693)')
#   plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.show()
#
# What to look for:
#   - Both curves dropping and staying close  → good fit
#   - Train keeps dropping, val starts rising → overfitting (stop earlier)
#   - Both stuck near 0.693                    → network isn't learning

---

## Phase 3: Compare and Reflect

Evaluate the neural network on the test set and compare to the baseline.

Key questions:
- Did the MLP beat logistic regression?
- How much more code did you write?
- How much longer did debugging take?
- Was the added complexity justified?

In [ ]:
# Cell 16: Evaluate MLP and compare
# TODO: Your AI will help you write this
#
# model.eval()
# with torch.no_grad():
#     test_outputs = model(X_test_tensor)
#     nn_pred = test_outputs.argmax(dim=1).numpy()
#
# nn_acc = accuracy_score(y_test, nn_pred)
# nn_f1 = f1_score(y_test, nn_pred)
#
# print('=== Model Comparison ===')
# print(f'LogReg:  Accuracy={lr_acc:.3f}, F1={lr_f1:.3f}')
# print(f'MLP:     Accuracy={nn_acc:.3f}, F1={nn_f1:.3f}')


In [ ]:
# Cell 17: Confusion matrices
# TODO: Plot side-by-side confusion matrices for LogReg and MLP


---

## Was It Worth It?

Write your honest assessment here:
- How much did the MLP improve over LogReg (if at all)?
- How much more code and debugging effort did the MLP require?
- Would you recommend using a neural network for this dataset? Why or why not?
- What did you learn from the experience?

---

## Bonus Challenges

See the [challenge brief](README.md) for 7 bonus challenges including:
- Learning rate sweep
- Architecture search
- Dropout experiment
- Channel selection
- Early stopping